1. Tự động đọc dữ liệu geojson từ nguồn online --> tên tỉnh --> huyện --> xã (vị trí địa lý Geometry) --> lưu vào file vietnam_communes_epsg4326.geojson
- Kiểm tra & Chuẩn hóa Hệ quy chiếu (CRS) về EPSG:4326

In [ ]:
import os
import zipfile
import requests
import geopandas as gpd
from shapely.validation import make_valid

# -------------------------------------------------------------
# 1. Cấu hình đường dẫn & URL GADM 4.1 cho Việt Nam
# -------------------------------------------------------------
DATA_DIR = "./data_spatial"
ZIP_PATH = os.path.join(DATA_DIR, "gadm41_VNM_shp.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "gadm41_VNM")
OUTPUT_CLEAN_PATH = os.path.join(DATA_DIR, "vietnam_communes_epsg4326.geojson")

os.makedirs(DATA_DIR, exist_ok=True)

# URL chính thức của GADM 4.1 cho gói Shapefile Việt Nam
GADM_VNM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_VNM_shp.zip"

# -------------------------------------------------------------
# 2. Tải và giải nén dữ liệu
# -------------------------------------------------------------
if not os.path.exists(ZIP_PATH):
    print(f"[*] Đang tải dữ liệu GADM từ: {GADM_VNM_URL}...")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(GADM_VNM_URL, headers=headers, stream=True)
    response.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    print("[+] Tải thành công.")
else:
    print("[*] File nén đã tồn tại, bỏ qua bước tải.")

if not os.path.exists(EXTRACT_DIR):
    print("[*] Đang giải nén file...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("[+] Giải nén hoàn tất.")

# -------------------------------------------------------------
# 3. Đọc dữ liệu Level 3 (Cấp Xã/Phường/Thị trấn)
# -------------------------------------------------------------
shp_level3_path = os.path.join(EXTRACT_DIR, "gadm41_VNM_3.shp")
print(f"[*] Đang nạp Shapefile cấp xã: {shp_level3_path}")
gdf_communes = gpd.read_file(shp_level3_path)

# -------------------------------------------------------------
# 4. Kiểm tra & Chuẩn hóa Hệ quy chiếu (CRS) về EPSG:4326
# -------------------------------------------------------------
target_crs = "EPSG:4326"

print(f"[*] CRS ban đầu: {gdf_communes.crs}")
if gdf_communes.crs is None:
    print("[!] Dữ liệu không có thông tin CRS, thiết lập mặc định EPSG:4326.")
    gdf_communes = gdf_communes.set_crs(target_crs)
elif gdf_communes.crs.to_string() != target_crs:
    print(f"[*] Chuyển đổi CRS từ {gdf_communes.crs} -> {target_crs}...")
    gdf_communes = gdf_communes.to_crs(target_crs)
else:
    print("[+] Dữ liệu đã ở định dạng chuẩn WGS 84 (EPSG:4326).")

# -------------------------------------------------------------
# 5. Sửa lỗi đa giác hình học (Geometry Cleaning & Validation)
# -------------------------------------------------------------
# Dữ liệu viễn thám dễ bị crash nếu đa giác xã bị tự cắt (self-intersection)
invalid_count = (~gdf_communes.is_valid).sum()
if invalid_count > 0:
    print(f"[!] Phát hiện {invalid_count} đa giác xã bị lỗi hình học. Đang tự động sửa...")
    gdf_communes["geometry"] = gdf_communes["geometry"].apply(make_valid)

# -------------------------------------------------------------
# 6. Kiểm tra cấu trúc thuộc tính và lưu file sạch
# -------------------------------------------------------------
cols_to_keep = ["GID_1", "NAME_1", "GID_2", "NAME_2", "GID_3", "NAME_3", "TYPE_3", "geometry"]
gdf_cleaned = gdf_communes[cols_to_keep].copy()

# Xuất ra GeoJSON chuẩn để dễ nạp lên GEE Asset hoặc chạy trên PyTorch
gdf_cleaned.to_file(OUTPUT_CLEAN_PATH, driver="GeoJSON")

print("\n--- KẾT QUẢ KIỂM TRA DỮ LIỆU ---")
print(f"- Tổng số đơn vị cấp xã: {len(gdf_cleaned):,}")
print(f"- Hệ tọa độ cuối cùng: {gdf_cleaned.crs}")
print(f"- Các cột định danh: {list(gdf_cleaned.columns[:-1])}")
print(f"- File lưu tại: {OUTPUT_CLEAN_PATH}")

In [ ]:
# Check
print(gdf_cleaned.shape)
print(gdf_cleaned.head(5))
print(gdf_cleaned[gdf_cleaned["GID_3"]== "VNM.27.3.1_1"])

2. KẾT NỐI GOOLE EARTH ENGINE

In [ ]:
import ee

# 1. Kích hoạt xác thực (nếu đã xác thực rồi thì cell sẽ tự bỏ qua)
try:
    ee.Initialize(project="angelic-digit-434804-r9")
    print("[+] GEE đã được khởi tạo thành công!")
except Exception as e:
    print("[*] Yêu cầu xác thực tài khoản Google...")
    ee.Authenticate()
    ee.Initialize(project="angelic-digit-434804-r9")
    print("[+] Xác thực và khởi tạo GEE hoàn tất!")

3. Lấy ảnh các xã của 1 tỉnh và lưu vào google drive folder


In [ ]:
import calendar
from datetime import datetime, timedelta
import os
import re
import unicodedata
import ee

PROJECT_ID = "angelic-digit-434804-r9"
ASSET_ID = f"projects/{PROJECT_ID}/assets/vietnam_communes_l3"

ee.Initialize(project=PROJECT_ID)
communes_fc = ee.FeatureCollection(ASSET_ID)


def normalize_str(s):
  if not s:
    return ""
  s = unicodedata.normalize("NFD", str(s))
  s = re.sub(r"[\u0300-\u036f]", "", s)
  s = s.replace("đ", "d").replace("Đ", "d")
  s = re.sub(r"[^a-zA-Z0-9]+", "_", s)
  return s.strip("_").lower()


# 1. Hàm tiền xử lý lọc mây pixel-level
def mask_s2_clean(img):
  qa = img.select("QA60")
  cloud_mask = (qa.bitwiseAnd(1 << 10).eq(0)).And(qa.bitwiseAnd(1 << 11).eq(0))
  return (
      img.updateMask(cloud_mask)
      .divide(10000)
      .select(
          ["B2", "B3", "B4", "B8", "B11", "B12"],
          ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"],
      )
  )


# 2. Hàm tính 4 chỉ số phổ chuyên dụng
def add_indices(img):
  ndvi = img.normalizedDifference(["NIR", "RED"]).rename("NDVI")
  ndbi = img.normalizedDifference(["SWIR1", "NIR"]).rename("NDBI")
  mndwi = img.normalizedDifference(["GREEN", "SWIR1"]).rename("MNDWI")
  bsi = img.expression(
      "((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))",
      {
          "SWIR1": img.select("SWIR1"),
          "RED": img.select("RED"),
          "NIR": img.select("NIR"),
          "BLUE": img.select("BLUE"),
      },
  ).rename("BSI")
  return img.addBands([ndvi, ndbi, mndwi, bsi])


# 3. Hàm tổng hợp ảnh thích ứng tự động (Temporal Window Expansion)
def get_adaptive_monthly_composite(year, month, commune_geom, collection_id):
  _, last_day = calendar.monthrange(year, month)
  dt_start = datetime(year, month, 1)
  dt_end = datetime(year, month, last_day)

  s_date = dt_start.strftime("%Y-%m-%d")
  e_date = dt_end.strftime("%Y-%m-%d")

  col = (
      ee.ImageCollection(collection_id)
      .filterBounds(commune_geom)
      .filterDate(s_date, e_date)
  )

  if col.size().getInfo() == 0:
    print(
        f"  [!] Tháng {month:02d}/{year} không có ảnh. Nới rộng biên (+/- 15"
        " ngày)..."
    )
    exp_start = (dt_start - timedelta(days=15)).strftime("%Y-%m-%d")
    exp_end = (dt_end + timedelta(days=15)).strftime("%Y-%m-%d")

    col = (
        ee.ImageCollection(collection_id)
        .filterBounds(commune_geom)
        .filterDate(exp_start, exp_end)
    )

    if col.size().getInfo() == 0:
      print(f"  [!] Tiếp tục nới rộng biên (+/- 30 ngày)...")
      exp_start_max = (dt_start - timedelta(days=30)).strftime("%Y-%m-%d")
      exp_end_max = (dt_end + timedelta(days=30)).strftime("%Y-%m-%d")

      col = (
          ee.ImageCollection(collection_id)
          .filterBounds(commune_geom)
          .filterDate(exp_start_max, exp_end_max)
      )

  if col.size().getInfo() > 0:
    return col.map(mask_s2_clean).median()
  return None


# 4. Trích xuất danh mục Tỉnh (NAME_1 <-> GID_1)
prov_map = {}
if "gdf_cleaned" in globals() and "NAME_1" in gdf_cleaned.columns:
  print("[*] Đang đọc danh mục tỉnh từ 'gdf_cleaned'...")
  unique_provs = (
      gdf_cleaned[["NAME_1", "GID_1"]].drop_duplicates().dropna().values
  )
  for name, gid1 in unique_provs:
    prov_map[str(name)] = str(gid1)
else:
  print("[*] Đang đọc danh mục tỉnh từ Asset GEE...")
  sample_props = communes_fc.first().getInfo()["properties"]
  col_pname = "NAME_1" if "NAME_1" in sample_props else None
  if col_pname:
    data = communes_fc.reduceColumns(
        ee.Reducer.toList(2), [col_pname, "GID_3"]
    ).getInfo()["list"]
    for name, gid3 in data:
      if name and gid3:
        parts = str(gid3).split(".")
        if len(parts) >= 2:
          gid1 = f"{parts[0]}.{parts[1]}_1"
          prov_map[str(name)] = gid1

if not prov_map:
  gid3_list = communes_fc.aggregate_array("GID_3").getInfo()
  distinct_gids = sorted(
      list(
          set(
              [
                  f"{x.split('.')[0]}.{x.split('.')[1]}_1"
                  for x in gid3_list
                  if "." in str(x)
              ]
          )
      )
  )
  for gid in distinct_gids:
    prov_map[gid] = gid

# 5. Hiển thị danh mục Tỉnh
print("=" * 88)
print(f"[+] DANH MỤC {len(prov_map)} TỈNH/THÀNH PHỐ KÈM MÃ PROVINCE:")
items = [f"{name} ({gid})" for name, gid in sorted(prov_map.items())]
for i in range(0, len(items), 2):
  row_items = items[i : i + 2]
  print("  ".join(f"{item:<42}" for item in row_items))
print("=" * 88)

# 6. Chọn Tỉnh
user_prov = input("\n👉 1. Nhập TÊN hoặc MÃ tỉnh/thành phố: ").strip()
norm_prov = normalize_str(user_prov)

matched_prov_name = None
for official_name, gid in prov_map.items():
  if (
      normalize_str(official_name) == norm_prov
      or normalize_str(gid) == norm_prov
  ):
    matched_prov_name = official_name
    break

if not matched_prov_name:
  print(f"\n[-] LỖI: Không tìm thấy tỉnh nào khớp với '{user_prov}'.")
else:
  target_prov_gid = prov_map[matched_prov_name]
  print(f"[+] Đã chọn tỉnh: '{matched_prov_name}' (Mã: {target_prov_gid})")

  # 7. Tải danh sách xã
  print(f"[*] Đang tải danh sách xã thuộc {matched_prov_name}...")
  prov_prefix = target_prov_gid.replace("_1", "") + "."
  commune_map = {}

  if "gdf_cleaned" in globals() and "NAME_3" in gdf_cleaned.columns:
    df_sub = gdf_cleaned[gdf_cleaned["GID_3"].str.startswith(prov_prefix)]
    for _, row in df_sub.iterrows():
      gid3 = str(row["GID_3"])
      c_type = str(row.get("TYPE_3", "")).strip()
      c_name = str(row["NAME_3"]).strip()
      full_name = (
          f"{c_type} {c_name}"
          if c_type and c_type.lower() != "nan"
          else c_name
      )
      commune_map[gid3] = full_name
  else:
    prov_communes = communes_fc.filter(
        ee.Filter.stringStartsWith("GID_3", prov_prefix)
    )
    first_feat = prov_communes.first().getInfo()
    available_cols = (
        list(first_feat["properties"].keys()) if first_feat else []
    )

    possible_name_cols = [
        "NAME_3",
        "NAME_3_vi",
        "VARNAME_3",
        "ADM3_VI",
        "NAME_2",
    ]
    name_col = next(
        (col for col in possible_name_cols if col in available_cols), None
    )
    type_col = "TYPE_3" if "TYPE_3" in available_cols else None

    features = prov_communes.getInfo()["features"]
    for f in features:
      props = f["properties"]
      gid3 = props.get("GID_3")
      c_name = props.get(name_col) if name_col else None
      c_type = props.get(type_col, "") if type_col else ""
      full_name = (
          f"{c_type} {c_name}".strip() if c_name and c_type else (c_name or gid3)
      )
      commune_map[gid3] = full_name

  print("=" * 88)
  print(
      f"[+] DANH SÁCH {len(commune_map)} XÃ/PHƯỜNG THUỘC"
      f" {matched_prov_name.upper()}:"
  )
  c_items = [
      f"{c_name} ({gid})"
      for gid, c_name in sorted(commune_map.items(), key=lambda x: x[1])
  ]
  for i in range(0, len(c_items), 2):
    row_items = c_items[i : i + 2]
    print("  ".join(f"{item:<42}" for item in row_items))
  print("=" * 88)

  # 8. Người dùng chọn Xã
  user_commune = input(
      "\n👉 2. Nhập TÊN hoặc MÃ XÃ (GID_3) cần trích xuất: "
  ).strip()
  norm_commune = normalize_str(user_commune)

  matched_gid3 = None
  matched_cname = None
  for gid, c_name in commune_map.items():
    if (
        normalize_str(c_name) == norm_commune
        or normalize_str(gid) == norm_commune
    ):
      matched_gid3 = gid
      matched_cname = c_name
      break

  if not matched_gid3:
    print(f"\n[-] LỖI: Không tìm thấy xã nào khớp với '{user_commune}'.")
  else:
    print(f"[+] XÁC NHẬN: {matched_cname} (Mã: {matched_gid3})")

    commune_geom = communes_fc.filter(
        ee.Filter.eq("GID_3", matched_gid3)
    ).geometry()

    # Chuẩn hóa tên và mã cho quy tắc đặt tên thư mục & file
    clean_pname = normalize_str(matched_prov_name)
    clean_cname = normalize_str(matched_cname)
    safe_gid3 = matched_gid3.replace(".", "_")

    start_year = 2016
    start_month = 1
    current_year = datetime.now().year
    current_month = datetime.now().month

    time_start_str = f"{start_year}{start_month:02d}"
    time_end_str = f"{current_year}{current_month:02d}"

    # QUY TẮC ĐẶT TÊN THƯ MỤC: S2_Day_têntỉnh_tênxã_mãxã_thờigianbắtđầu-thờigiankếtthúc
    drive_folder = f"S2_Day_{clean_pname}_{clean_cname}_{safe_gid3}_{time_start_str}-{time_end_str}"

    print("\n" + "=" * 80)
    print(
        f"[*] THIẾT LẬP XUẤT CHUỖI ẢNH ({time_start_str} -> {time_end_str}):"
    )
    print(f"    - Thư mục Google Drive: {drive_folder}")
    print(
        f"    - Định dạng tên file:   S2_Day_{clean_pname}_{clean_cname}_{safe_gid3}_[YYYYMM]"
    )
    print(f"    - Độ phân giải ảnh:     20m/pixel (10 kênh đa phổ)")
    print("=" * 80)

    tasks_submitted = 0

    for yr in range(start_year, current_year + 1):
      end_m = current_month if yr == current_year else 12
      for m in range(1, end_m + 1):
        time_point_str = f"{yr}{m:02d}"

        collection_id = (
            "COPERNICUS/S2_SR_HARMONIZED"
            if yr >= 2019
            else "COPERNICUS/S2_HARMONIZED"
        )

        composite = get_adaptive_monthly_composite(
            yr, m, commune_geom, collection_id
        )

        if composite is not None:
          export_img = add_indices(composite).clip(commune_geom)

          # QUY TẮC ĐẶT TÊN FILE: S2_Day_têntỉnh_tênxã_mãxã_thờigianlấyảnh
          file_pfx = f"S2_Day_{clean_pname}_{clean_cname}_{safe_gid3}_{time_point_str}"
          task_desc = f"TS_{safe_gid3}_{time_point_str}"

          task = ee.batch.Export.image.toDrive(
              image=export_img,
              description=task_desc,
              folder=drive_folder,
              fileNamePrefix=file_pfx,
              region=commune_geom,
              scale=20,
              crs="EPSG:4326",
              maxPixels=1e9,
          )
          task.start()
          tasks_submitted += 1
          print(f"[+] Đã gửi Task: Tháng {m:02d}/{yr} -> File: {file_pfx}")
        else:
          print(
              f"[-] Bỏ qua tháng {m:02d}/{yr}: Không tìm thấy ảnh ngay cả khi"
              " nới biên tối đa."
          )

    print("\n" + "=" * 80)
    print(
        f"[+] HOÀN TẤT! Đã gửi tổng cộng {tasks_submitted} tác vụ về Drive thư"
        f" mục:\n    👉 Drive/{drive_folder}"
    )
    print("=" * 80)

In [ ]:
import calendar
from datetime import datetime, timedelta
import os
import re
import unicodedata
import ee

PROJECT_ID = "angelic-digit-434804-r9"
ASSET_ID = f"projects/{PROJECT_ID}/assets/vietnam_communes_l3"

ee.Initialize(project=PROJECT_ID)
communes_fc = ee.FeatureCollection(ASSET_ID)


def normalize_str(s):
  if not s:
    return ""
  s = unicodedata.normalize("NFD", str(s))
  s = re.sub(r"[\u0300-\u036f]", "", s)
  s = s.replace("đ", "d").replace("Đ", "d")
  s = re.sub(r"[^a-zA-Z0-9]+", "_", s)
  return s.strip("_").lower()


# Hàm sắp xếp tự nhiên theo mã số (đảm bảo .2_1 đứng trước .10_1)
def natural_sort_key(gid_str):
  return [
      int(text) if text.isdigit() else text.lower()
      for text in re.split(r"(\d+)", str(gid_str))
  ]


# 1. Tiền xử lý lọc mây pixel-level
def mask_s2_clean(img):
  qa = img.select("QA60")
  cloud_mask = (qa.bitwiseAnd(1 << 10).eq(0)).And(qa.bitwiseAnd(1 << 11).eq(0))
  return (
      img.updateMask(cloud_mask)
      .divide(10000)
      .select(
          ["B2", "B3", "B4", "B8", "B11", "B12"],
          ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"],
      )
  )


# 2. Tính 4 chỉ số phổ chuyên dụng
def add_indices(img):
  ndvi = img.normalizedDifference(["NIR", "RED"]).rename("NDVI")
  ndbi = img.normalizedDifference(["SWIR1", "NIR"]).rename("NDBI")
  mndwi = img.normalizedDifference(["GREEN", "SWIR1"]).rename("MNDWI")
  bsi = img.expression(
      "((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))",
      {
          "SWIR1": img.select("SWIR1"),
          "RED": img.select("RED"),
          "NIR": img.select("NIR"),
          "BLUE": img.select("BLUE"),
      },
  ).rename("BSI")
  return img.addBands([ndvi, ndbi, mndwi, bsi])


# 3. Tổng hợp ảnh thích ứng tự động (Temporal Window Expansion)
def get_adaptive_monthly_composite(year, month, commune_geom, collection_id):
  _, last_day = calendar.monthrange(year, month)
  dt_start = datetime(year, month, 1)
  dt_end = datetime(year, month, last_day)

  s_date = dt_start.strftime("%Y-%m-%d")
  e_date = dt_end.strftime("%Y-%m-%d")

  col = (
      ee.ImageCollection(collection_id)
      .filterBounds(commune_geom)
      .filterDate(s_date, e_date)
  )

  if col.size().getInfo() == 0:
    print(
        f"  [!] Tháng {month:02d}/{year} không có ảnh. Nới rộng biên (+/- 15"
        " ngày)..."
    )
    exp_start = (dt_start - timedelta(days=15)).strftime("%Y-%m-%d")
    exp_end = (dt_end + timedelta(days=15)).strftime("%Y-%m-%d")

    col = (
        ee.ImageCollection(collection_id)
        .filterBounds(commune_geom)
        .filterDate(exp_start, exp_end)
    )

    if col.size().getInfo() == 0:
      print(f"  [!] Tiếp tục nới rộng biên (+/- 30 ngày)...")
      exp_start_max = (dt_start - timedelta(days=30)).strftime("%Y-%m-%d")
      exp_end_max = (dt_end + timedelta(days=30)).strftime("%Y-%m-%d")

      col = (
          ee.ImageCollection(collection_id)
          .filterBounds(commune_geom)
          .filterDate(exp_start_max, exp_end_max)
      )

  if col.size().getInfo() > 0:
    return col.map(mask_s2_clean).median()
  return None


# 4. Trích xuất danh mục Tỉnh (NAME_1 <-> GID_1)
prov_map = {}
if "gdf_cleaned" in globals() and "NAME_1" in gdf_cleaned.columns:
  print("[*] Đang đọc danh mục tỉnh từ 'gdf_cleaned'...")
  unique_provs = (
      gdf_cleaned[["NAME_1", "GID_1"]].drop_duplicates().dropna().values
  )
  for name, gid1 in unique_provs:
    prov_map[str(name)] = str(gid1)
else:
  print("[*] Đang đọc danh mục tỉnh từ Asset GEE...")
  sample_props = communes_fc.first().getInfo()["properties"]
  col_pname = "NAME_1" if "NAME_1" in sample_props else None
  if col_pname:
    data = communes_fc.reduceColumns(
        ee.Reducer.toList(2), [col_pname, "GID_3"]
    ).getInfo()["list"]
    for name, gid3 in data:
      if name and gid3:
        parts = str(gid3).split(".")
        if len(parts) >= 2:
          gid1 = f"{parts[0]}.{parts[1]}_1"
          prov_map[str(name)] = gid1

if not prov_map:
  gid3_list = communes_fc.aggregate_array("GID_3").getInfo()
  distinct_gids = sorted(
      list(
          set(
              [
                  f"{x.split('.')[0]}.{x.split('.')[1]}_1"
                  for x in gid3_list
                  if "." in str(x)
              ]
          )
      )
  )
  for gid in distinct_gids:
    prov_map[gid] = gid

# 5. Hiển thị danh mục Tỉnh
print("=" * 88)
print(f"[+] DANH MỤC {len(prov_map)} TỈNH/THÀNH PHỐ KÈM MÃ PROVINCE:")
items = [f"{name} ({gid})" for name, gid in sorted(prov_map.items())]
for i in range(0, len(items), 2):
  row_items = items[i : i + 2]
  print("  ".join(f"{item:<42}" for item in row_items))
print("=" * 88)

# 6. Người dùng chọn Tỉnh
user_prov = input("\n👉 1. Nhập TÊN hoặc MÃ tỉnh/thành phố: ").strip()
norm_prov = normalize_str(user_prov)

matched_prov_name = None
for official_name, gid in prov_map.items():
  if (
      normalize_str(official_name) == norm_prov
      or normalize_str(gid) == norm_prov
  ):
    matched_prov_name = official_name
    break

if not matched_prov_name:
  print(f"\n[-] LỖI: Không tìm thấy tỉnh nào khớp với '{user_prov}'.")
else:
  target_prov_gid = prov_map[matched_prov_name]
  print(f"[+] Đã chọn tỉnh: '{matched_prov_name}' (Mã: {target_prov_gid})")

  # 7. Tải danh sách xã
  print(f"[*] Đang tải danh sách xã thuộc {matched_prov_name}...")
  prov_prefix = target_prov_gid.replace("_1", "") + "."
  commune_map = {}

  if "gdf_cleaned" in globals() and "NAME_3" in gdf_cleaned.columns:
    df_sub = gdf_cleaned[gdf_cleaned["GID_3"].str.startswith(prov_prefix)]
    for _, row in df_sub.iterrows():
      gid3 = str(row["GID_3"])
      c_type = str(row.get("TYPE_3", "")).strip()
      c_name = str(row["NAME_3"]).strip()
      full_name = (
          f"{c_type} {c_name}"
          if c_type and c_type.lower() != "nan"
          else c_name
      )
      commune_map[gid3] = full_name
  else:
    prov_communes = communes_fc.filter(
        ee.Filter.stringStartsWith("GID_3", prov_prefix)
    )
    first_feat = prov_communes.first().getInfo()
    available_cols = (
        list(first_feat["properties"].keys()) if first_feat else []
    )

    possible_name_cols = [
        "NAME_3",
        "NAME_3_vi",
        "VARNAME_3",
        "ADM3_VI",
        "NAME_2",
    ]
    name_col = next(
        (col for col in possible_name_cols if col in available_cols), None
    )
    type_col = "TYPE_3" if "TYPE_3" in available_cols else None

    features = prov_communes.getInfo()["features"]
    for f in features:
      props = f["properties"]
      gid3 = props.get("GID_3")
      c_name = props.get(name_col) if name_col else None
      c_type = props.get(type_col, "") if type_col else ""
      full_name = (
          f"{c_type} {c_name}".strip() if c_name and c_type else (c_name or gid3)
      )
      commune_map[gid3] = full_name

  # SẮP XẾP THEO MÃ XÃ (GID_3)
  sorted_by_gid = sorted(commune_map.items(), key=lambda x: natural_sort_key(x[0]))

  print("=" * 88)
  print(
      f"[+] DANH SÁCH {len(commune_map)} XÃ/PHƯỜNG THUỘC"
      f" {matched_prov_name.upper()} (ĐÃ ORDER THEO MÃ GID_3):"
  )
  c_items = [f"{c_name} ({gid})" for gid, c_name in sorted_by_gid]
  for i in range(0, len(c_items), 2):
    row_items = c_items[i : i + 2]
    print("  ".join(f"{item:<42}" for item in row_items))
  print("=" * 88)

  # 8. Người dùng chọn Xã
  user_commune = input(
      "\n👉 2. Nhập TÊN hoặc MÃ XÃ (GID_3) cần trích xuất: "
  ).strip()
  norm_commune = normalize_str(user_commune)

  matched_gid3 = None
  matched_cname = None
  for gid, c_name in commune_map.items():
    if (
        normalize_str(c_name) == norm_commune
        or normalize_str(gid) == norm_commune
    ):
      matched_gid3 = gid
      matched_cname = c_name
      break

  if not matched_gid3:
    print(f"\n[-] LỖI: Không tìm thấy xã nào khớp với '{user_commune}'.")
  else:
    print(f"[+] XÁC NHẬN: {matched_cname} (Mã: {matched_gid3})")

    commune_geom = communes_fc.filter(
        ee.Filter.eq("GID_3", matched_gid3)
    ).geometry()

    clean_pname = normalize_str(matched_prov_name)
    clean_cname = normalize_str(matched_cname)
    safe_gid3 = matched_gid3.replace(".", "_")

    start_year = 2016
    start_month = 1
    current_year = datetime.now().year
    current_month = datetime.now().month

    time_start_str = f"{start_year}{start_month:02d}"
    time_end_str = f"{current_year}{current_month:02d}"

    # QUY TẮC ĐẶT TÊN THƯ MỤC
    drive_folder = f"S2_Day_{clean_pname}_{clean_cname}_{safe_gid3}_{time_start_str}-{time_end_str}"

    print("\n" + "=" * 80)
    print(
        f"[*] THIẾT LẬP XUẤT CHUỖI ẢNH ({time_start_str} -> {time_end_str}):"
    )
    print(f"    - Thư mục Google Drive: {drive_folder}")
    print(
        f"    - Quy chuẩn tên file:   S2_Day_{clean_pname}_{clean_cname}_{safe_gid3}_[YYYYMM]"
    )
    print(f"    - Độ phân giải ảnh:     20m/pixel (10 kênh đa phổ)")
    print("=" * 80)

    tasks_submitted = 0

    for yr in range(start_year, current_year + 1):
      end_m = current_month if yr == current_year else 12
      for m in range(1, end_m + 1):
        time_point_str = f"{yr}{m:02d}"

        collection_id = (
            "COPERNICUS/S2_SR_HARMONIZED"
            if yr >= 2019
            else "COPERNICUS/S2_HARMONIZED"
        )

        composite = get_adaptive_monthly_composite(
            yr, m, commune_geom, collection_id
        )

        if composite is not None:
          export_img = add_indices(composite).clip(commune_geom)

          # QUY TẮC ĐẶT TÊN FILE
          file_pfx = f"S2_Day_{clean_pname}_{clean_cname}_{safe_gid3}_{time_point_str}"
          task_desc = f"TS_{safe_gid3}_{time_point_str}"

          task = ee.batch.Export.image.toDrive(
              image=export_img,
              description=task_desc,
              folder=drive_folder,
              fileNamePrefix=file_pfx,
              region=commune_geom,
              scale=20,
              crs="EPSG:4326",
              maxPixels=1e9,
          )
          task.start()
          tasks_submitted += 1
          print(f"[+] Đã gửi Task: Tháng {m:02d}/{yr} -> File: {file_pfx}")
        else:
          print(
              f"[-] Bỏ qua tháng {m:02d}/{yr}: Không tìm thấy ảnh ngay cả khi"
              " nới biên tối đa."
          )

    print("\n" + "=" * 80)
    print(
        f"[+] HOÀN TẤT! Đã gửi tổng cộng {tasks_submitted} tác vụ về thư"
        f" mục:\n    👉 Drive/{drive_folder}"
    )
    print("=" * 80)

4. Đọc và show thử 1 ảnh .tif với Màu giả hồng ngoại (NIR-R-G) và màu thực RGB (kéo file ảnh vào bên trái colab)

In [ ]:
!pip install -q rasterio
#?

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyproj
import rasterio
from shapely.geometry import box
from shapely.ops import transform

# 1. Đường dẫn file ảnh tif bên trái Colab
image_path = "/content/s2_multiband_VNM_27_1_3_1_2020_03.tif"  # Thay bằng tên file thực tế của bạn


def stretch_band(band, lower_percent=2, upper_percent=98):
  """Co giãn tương phản động 2% - 98% giúp ảnh hiển thị sắc nét."""
  valid_pixels = band[~np.isnan(band) & (band > 0)]
  if len(valid_pixels) == 0:
    return band
  p_low, p_high = np.percentile(valid_pixels, (lower_percent, upper_percent))
  stretched = np.clip((band - p_low) / (p_high - p_low + 1e-6), 0, 1)
  return stretched


# 2. Đọc ảnh, thống kê pixel, độ phân giải và diện tích
with rasterio.open(image_path) as src:
  num_bands = src.count
  crs = src.crs
  transform_mat = src.transform
  width = src.width
  height = src.height
  total_pixels_raw = width * height

  red = src.read(1).astype(float)
  green = src.read(2).astype(float)
  blue = src.read(3).astype(float)
  nir = src.read(4).astype(float) if num_bands >= 4 else None

  # Mặt nạ lọc ranh giới thực của xã (loại NoData / viền đen)
  if src.nodata is not None:
    mask = red != src.nodata
  else:
    mask = (red > 0) | (green > 0) | (blue > 0)

  valid_commune_pixels = int(np.sum(mask))
  nodata_border_pixels = total_pixels_raw - valid_commune_pixels

  valid_ratio = (valid_commune_pixels / total_pixels_raw) * 100
  nodata_ratio = (nodata_border_pixels / total_pixels_raw) * 100

  # Tính toán độ phân giải không gian (GSD theo mét) và diện tích
  if crs.is_projected:
    res_x = abs(transform_mat[0])
    res_y = abs(transform_mat[4])
    pixel_area_m2 = res_x * res_y
  else:
    bounds = src.bounds
    center_lon = (bounds.left + bounds.right) / 2
    center_lat = (bounds.bottom + bounds.top) / 2
    utm_zone = int((center_lon + 180) / 6) + 1
    utm_crs = f"+proj=utm +zone={utm_zone} +datum=WGS84 +units=m +no_defs"

    project = pyproj.Transformer.from_crs(
        crs, utm_crs, always_xy=True
    ).transform
    geom_box = box(
        bounds.left,
        bounds.bottom,
        bounds.left + abs(transform_mat[0]),
        bounds.bottom + abs(transform_mat[4]),
    )
    pixel_area_m2 = transform(project, geom_box).area
    res_x = np.sqrt(pixel_area_m2)
    res_y = res_x

  total_area_km2 = (valid_commune_pixels * pixel_area_m2) / 1e6
  total_area_ha = (valid_commune_pixels * pixel_area_m2) / 1e4

# In chi tiết thông số
print("=" * 60)
print(f"[*] Tệp xử lý: {os.path.basename(image_path)}")
print(f"[+] Kích thước ma trận ảnh: {height} x {width} pixels (Cao x Rộng)")
print(f"[+] Số kênh quang phổ (Bands): {num_bands}")
print(f"[+] Độ phân giải không gian: ~{res_x:.1f}m x {res_y:.1f}m / pixel")
print("-" * 60)
print(
    f"[+] Tổng số pixel khung hình chữ nhật: {total_pixels_raw:,} pixels"
    " (100.00%)"
)
print(
    f"[+] Số pixel HỢP LỆ (thực tế thuộc xã): {valid_commune_pixels:,} pixels"
    f" ({valid_ratio:.2f}%)"
)
print(
    f"[+] Số pixel VIỀN ĐEN / NoData:        {nodata_border_pixels:,} pixels"
    f" ({nodata_ratio:.2f}%)"
)
print("-" * 60)
print(f"[+] Diện tích tính toán: {total_area_km2:.2f} km² ({total_area_ha:.2f} ha)")
print("=" * 60)

# 3. Co giãn tương phản và ghép kênh
r_norm = stretch_band(red)
g_norm = stretch_band(green)
b_norm = stretch_band(blue)

# Ảnh RGB (True Color)
rgb_arr = np.dstack((r_norm, g_norm, b_norm))
rgb_arr[~mask] = 0
rgb_uint8 = (rgb_arr * 255).astype(np.uint8)

# 4. Xuất file ảnh trực tiếp vào /content
base_name = os.path.splitext(os.path.basename(image_path))[0]
out_rgb_path = f"/content/{base_name}_RGB.png"
Image.fromarray(rgb_uint8).save(out_rgb_path)
print(f"[+] Đã lưu ảnh Màu thực (RGB): {out_rgb_path}")

out_nir_path = None
if nir is not None:
  nir_norm = stretch_band(nir)
  false_arr = np.dstack((nir_norm, r_norm, g_norm))
  false_arr[~mask] = 0
  false_uint8 = (false_arr * 255).astype(np.uint8)

  out_nir_path = f"/content/{base_name}_FalseColor_NIR.png"
  Image.fromarray(false_uint8).save(out_nir_path)
  print(f"[+] Đã lưu ảnh Màu giả (NIR): {out_nir_path}")

# 5. Hiển thị kiểm tra trên Notebook
fig, ax = plt.subplots(1, 2 if nir is not None else 1, figsize=(15, 7), dpi=150)
info_text = f"Độ phân giải: ~{res_x:.1f}m | Diện tích: {total_area_km2:.2f} km²"

if nir is not None:
  ax[0].imshow(rgb_uint8)
  ax[0].set_title(
      f"Màu thực (RGB)\n{info_text}", fontsize=11, fontweight="bold"
  )
  ax[0].axis("off")

  ax[1].imshow(false_uint8)
  ax[1].set_title(
      f"Màu giả hồng ngoại (NIR-R-G)\n({total_area_ha:.1f} ha)",
      fontsize=11,
      fontweight="bold",
  )
  ax[1].axis("off")
else:
  ax.imshow(rgb_uint8)
  ax.set_title(f"Màu thực (RGB)\n{info_text}", fontsize=11, fontweight="bold")
  ax.axis("off")

plt.tight_layout()
plt.show()

5.1. Detect đường xá

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from skimage.morphology import remove_small_objects, skeletonize

# 1. Đường dẫn tệp ảnh đa phổ của bạn bên trái Colab
image_path = "/content/s2_multiband_VNM_27_1_1_1_2020_03.tif"


def stretch_band(band, lower_percent=2, upper_percent=98):
  """Co giãn tương phản động phục vụ trực quan hóa."""
  valid = band[~np.isnan(band) & (band > 0)]
  if len(valid) == 0:
    return band
  p_low, p_high = np.percentile(valid, (lower_percent, upper_percent))
  return np.clip((band - p_low) / (p_high - p_low + 1e-6), 0, 1)


# 2. Đọc các kênh quang phổ
with rasterio.open(image_path) as src:
  # Thứ tự các kênh xuất ra từ GEE pipeline:
  # 1: BLUE, 2: GREEN, 3: RED, 4: NIR, 5: SWIR1, 6: SWIR2, 7: NDVI, 8: NDBI, 9: MNDWI, 10: BSI
  blue = src.read(1).astype(float)
  green = src.read(2).astype(float)
  red = src.read(3).astype(float)
  nir = src.read(4).astype(float)
  swir1 = src.read(5).astype(float)

  # Đọc sẵn các chỉ số nếu có trong 10 kênh, hoặc tự tính lại
  if src.count >= 9:
    ndvi = src.read(7).astype(float)
    ndbi = src.read(8).astype(float)
    mndwi = src.read(9).astype(float)
  else:
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndbi = (swir1 - nir) / (swir1 + nir + 1e-6)
    mndwi = (green - swir1) / (green + swir1 + 1e-6)

  # Tạo mặt nạ vùng xã hợp lệ (loại trừ NoData)
  if src.nodata is not None:
    commune_mask = red != src.nodata
  else:
    commune_mask = (red > 0) | (green > 0) | (blue > 0)

# 3. Lọc phổ cho bề mặt đường nhựa / bê tông
# Đường trục chính có:
# - NDVI thấp (không phải cây cỏ, rừng, ruộng)
# - MNDWI thấp (không phải sông hồ, ao nước)
# - NDBI từ mức trung bình đến cao (bề mặt nhân tạo / không thấm nước)
road_spectral_mask = (
    (ndvi < 0.25)
    & (mndwi < -0.05)
    & (ndbi > -0.10)
    & commune_mask
)

# Chuyển về dạng ảnh nhị phân uint8 để xử lý hình thái học
binary_road = (road_spectral_mask * 255).astype(np.uint8)

# 4. Lọc hình thái học để giữ lại các vệt dài (Linear Features) và loại khối rộng (Khu công nghiệp/khu dân cư nén)
# Kernel hình chữ nhật / đường thẳng theo các góc (ngang, dọc, chéo)
kernel_len = 5  # ~100m chiều dài
k_h = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))
k_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_len))

# Phép mở hình thái học (Morphological Open) theo các hướng trục để bắt vệt giao thông
open_h = cv2.morphologyEx(binary_road, cv2.MORPH_OPEN, k_h)
open_v = cv2.morphologyEx(binary_road, cv2.MORPH_OPEN, k_v)
linear_roads = cv2.bitwise_or(open_h, open_v)

# Khử nhiễu các chấm nhỏ lẻ (< 10 pixels liên thông)
clean_roads = remove_small_objects(linear_roads > 0, min_size=10)

# Rút xương trục giao thông (Road Centerline Skeletonization)
road_skeleton = skeletonize(clean_roads)

# 5. Thống kê sơ bộ mạng lưới đường
total_road_pixels = np.sum(clean_roads)
road_centerline_pixels = np.sum(road_skeleton)
est_road_km = (road_centerline_pixels * 19.3) / 1000.0  # Mỗi pixel ~19.3m

print("=" * 55)
print(f"[*] Tổng số pixel thuộc hành lang giao thông: {total_road_pixels:,} pixels")
print(f"[*] Số pixel tim đường trục chính:             {road_centerline_pixels:,} pixels")
print(f"[+] Ước tính tổng chiều dài đường trục chính:  ~{est_road_km:.2f} km")
print("=" * 55)

# 6. Trực quan hóa kết quả
r_norm = stretch_band(red)
g_norm = stretch_band(green)
b_norm = stretch_band(blue)
rgb_img = np.dstack((r_norm, g_norm, b_norm))
rgb_img[~commune_mask] = 0

# Tạo lớp phủ (Overlay) đường màu đỏ rực rỡ lên ảnh màu thực
overlay_img = rgb_img.copy()
# Đánh dấu các pixel tim đường bằng màu vàng/đỏ nổi bật
overlay_img[road_skeleton] = [1.0, 0.0, 0.0]  # Màu đỏ cho tim đường

fig, ax = plt.subplots(1, 3, figsize=(18, 6), dpi=150)

# Ảnh RGB gốc
ax[0].imshow(rgb_img)
ax[0].set_title("1. Ảnh RGB Màu Thực", fontsize=11, fontweight="bold")
ax[0].axis("off")

# Mặt nạ nhị phân giao thông bóc tách
ax[1].imshow(clean_roads, cmap="magma")
ax[1].set_title("2. Mặt nạ hành lang đường bộ", fontsize=11, fontweight="bold")
ax[1].axis("off")

# Trục giao thông phủ lên ảnh thực
ax[2].imshow(overlay_img)
ax[2].set_title(
    f"3. Tim đường trục chính (Đỏ) - Ước tính: {est_road_km:.2f} km",
    fontsize=11,
    fontweight="bold",
)
ax[2].axis("off")

plt.tight_layout()
plt.show()

# 7. Tùy chọn lưu mặt nạ ra file png bên trái Colab
cv2.imwrite(
    f"/content/{os.path.splitext(os.path.basename(image_path))[0]}_Roads_Detected.png",
    (clean_roads * 255).astype(np.uint8),
)
print("[+] Đã lưu mặt nạ giao thông vào panel bên trái Colab!")

5.2 - Lấy thông tin giao thông và vẽ đường xá

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pyproj
import rasterio
import requests
from shapely.geometry import LineString, box
import geopandas as gpd

# 1. Đường dẫn file ảnh
image_path = "/content/s2_multiband_VNM_27_1_1_1_2020_03.tif"

def stretch_band(band, lower_percent=2, upper_percent=98):
    valid = band[~np.isnan(band) & (band > 0)]
    if len(valid) == 0:
        return band
    p_low, p_high = np.percentile(valid, (lower_percent, upper_percent))
    return np.clip((band - p_low) / (p_high - p_low + 1e-6), 0, 1)

# 2. Đọc tọa độ từ ảnh
with rasterio.open(image_path) as src:
    bounds = src.bounds
    crs = src.crs
    red = src.read(3).astype(float)
    green = src.read(2).astype(float)
    blue = src.read(1).astype(float)
    mask = (red != src.nodata) if src.nodata is not None else ((red > 0) | (green > 0) | (blue > 0))

    if crs and crs.to_epsg() != 4326:
        transformer = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        west, south = transformer.transform(bounds.left, bounds.bottom)
        east, north = transformer.transform(bounds.right, bounds.top)
    else:
        west, south, east, north = bounds.left, bounds.bottom, bounds.right, bounds.top

# 3. Danh sách các máy chủ Overpass Mirror dự phòng
mirrors = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter"
]

query = f"""
[out:json][timeout:25];
(
  way["highway"]({south},{west},{north},{east});
);
out geom;
"""

headers = {"User-Agent": "ResearchSpatialEcon/1.0 (contact@research.edu)"}
elements = []

for url in mirrors:
    print(f"[*] Thử kết nối đến mirror: {url} ...")
    try:
        resp = requests.post(url, data={"data": query}, headers=headers, timeout=20)
        if resp.status_code == 200:
            elements = resp.json().get("elements", [])
            print(f"[+] Kết nối THÀNH CÔNG! Đã tải về {len(elements)} đoạn đường.")
            break
        else:
            print(f"[-] Server phản hồi mã: {resp.status_code}, thử mirror tiếp theo...")
    except Exception as e:
        print(f"[-] Bị từ chối kết nối, chuyển mirror...")

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import shapes
from shapely.geometry import LineString, shape

# --- Tiếp nối sau khi đã tải xong danh sách 'elements' từ Overpass ---

lines, names, types = [], [], []
for el in elements:
  if "geometry" in el and len(el["geometry"]) >= 2:
    pts = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]
    lines.append(LineString(pts))
    names.append(el.get("tags", {}).get("name", "Đường không tên"))
    types.append(el.get("tags", {}).get("highway", "residential"))

if len(lines) > 0:
  gdf_roads = gpd.GeoDataFrame(
      {"name": names, "highway": types, "geometry": lines}, crs="EPSG:4326"
  )

  # 1. Trích xuất đa giác ranh giới thực của xã từ mặt nạ pixel (loại bỏ vùng đen NoData)
  mask_uint8 = mask.astype(np.uint8)
  commune_polygons = [
      shape(geom)
      for geom, val in shapes(mask_uint8, transform=src.transform)
      if val == 1
  ]
  commune_boundary = gpd.GeoDataFrame(
      geometry=commune_polygons, crs=src.crs
  ).unary_union

  # Đảm bảo ranh giới cùng hệ WGS84
  if src.crs and src.crs.to_epsg() != 4326:
    commune_boundary = (
        gpd.GeoSeries([commune_boundary], crs=src.crs)
        .to_crs("EPSG:4326")
        .iloc[0]
    )

  # Cắt chỉ giữ lại các đoạn đường nằm TRONG ranh giới thực của xã
  gdf_roads = gdf_roads.clip(commune_boundary)

  # 2. Định nghĩa trọng số hiển thị theo phân cấp đường (Hierarchy Styling)
  style_map = {
      # Trục huyết mạch: Nét lớn, màu đỏ/cam
      "motorway": {"width": 4.2, "color": "#FF2A00", "label": "Cao tốc"},
      "trunk": {"width": 4.0, "color": "#FF4500", "label": "Quốc lộ"},
      "primary": {"width": 3.5, "color": "#FF8C00", "label": "Đại lộ/Tỉnh lộ"},
      # Trục liên kết: Nét vừa, màu vàng rực
      "secondary": {
          "width": 2.8,
          "color": "#FFD700",
          "label": "Đường liên huyện",
      },
      "tertiary": {"width": 2.4, "color": "#FFFF00", "label": "Đường liên xã"},
      # Đường địa phương/ngõ: Nét mảnh, màu xám nhạt/trắng
      "residential": {"width": 1.8, "color": "#FFFFFF", "label": "Đường ngõ"},
      "unclassified": {
          "width": 1.8,
          "color": "#D3D3D3",
          "label": "Đường nội bộ",
      },
      "service": {"width": 1.5, "color": "#A9A9A9", "label": "Đường phụ"},
  }

  # Áp dụng cấu hình mặc định nếu gặp loại đường khác
  default_style = {"width": 0.8, "color": "#FFFFFF", "label": "Khác"}
  gdf_roads["linewidth"] = gdf_roads["highway"].apply(
      lambda x: style_map.get(x, default_style)["width"]
  )
  gdf_roads["color"] = gdf_roads["highway"].apply(
      lambda x: style_map.get(x, default_style)["color"]
  )

  # 3. Tính tổng chiều dài thực tế trong xã (UTM Zone 48N)
  gdf_utm = gdf_roads.to_crs("EPSG:32648")
  total_commune_km = gdf_utm.geometry.length.sum() / 1000.0

  print("=" * 60)
  print(f"[+] Số đoạn đường hợp lệ trong xã: {len(gdf_roads)} đoạn")
  print(f"[+] Tổng chiều dài đường trong xã: {total_commune_km:.2f} km")
  print("=" * 60)
else:
  gdf_roads = None
  total_commune_km = 0.0

# 4. Trực quan hóa chuẩn xác
r_norm = stretch_band(red)
g_norm = stretch_band(green)
b_norm = stretch_band(blue)
rgb_img = np.dstack((r_norm, g_norm, b_norm))
rgb_img[~mask] = 0

fig, ax = plt.subplots(figsize=(11, 11), dpi=160)
ax.imshow(rgb_img, extent=[west, east, south, north], origin="upper")

if gdf_roads is not None and not gdf_roads.empty:
  # Vẽ từng loại đường theo độ dày và màu sắc tương ứng
  for hw_type, group in gdf_roads.groupby("highway"):
    style = style_map.get(hw_type, default_style)
    group.plot(
        ax=ax,
        linewidth=style["width"],
        edgecolor=style["color"],
        alpha=0.9,
        label=style["label"],
        zorder=3 if style["width"] > 1.5 else 2,
    )

  ax.set_title(
      "Mạng lưới giao thông phân cấp theo lộ giới thực tế\n"
      f"(Chỉ trong phạm vi xã - Tổng chiều dài: {total_commune_km:.2f} km)",
      fontsize=12,
      fontweight="bold",
  )
else:
  ax.set_title("Không có dữ liệu đường", fontsize=12)

ax.set_xlabel("Kinh độ (Longitude)")
ax.set_ylabel("Vĩ độ (Latitude)")
plt.tight_layout()
plt.show()

6. Khu công nghiệp, nhà máy, kho bãi logistics quy mô lớn: Kênh nhận diện: Kênh SWIR1 và SWIR2 cực kỳ nhạy với vật liệu kim loại (mái tôn lợp xưởng) và bề mặt bê tông công nghiệp, tạo nên tín hiệu phản xạ trắng/sáng rõ rệt tách biệt hoàn toàn khỏi khu dân cư thông thường.

In [ ]:
import os
import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from scipy.ndimage import generic_filter
from skimage.measure import label, regionprops

# 1. Đường dẫn file ảnh
image_path = "/content/s2_multiband_VNM_27_1_1_1_2020_03.tif"


def stretch_band(band, lower_percent=2, upper_percent=98):
  valid = band[~np.isnan(band) & (band > 0)]
  if len(valid) == 0:
    return band
  p_low, p_high = np.percentile(valid, (lower_percent, upper_percent))
  return np.clip((band - p_low) / (p_high - p_low + 1e-6), 0, 1)


# 2. Đọc kênh ảnh
with rasterio.open(image_path) as src:
  red = src.read(3).astype(float)
  green = src.read(2).astype(float)
  blue = src.read(1).astype(float)
  nir = src.read(4).astype(float)
  swir1 = src.read(5).astype(float)
  swir2 = src.read(6).astype(float)

  if src.nodata is not None:
    commune_mask = red != src.nodata
  else:
    commune_mask = (red > 0) | (green > 0) | (blue > 0)

# 3. Tính toán quang phổ kim loại
ndvi = (nir - red) / (nir + red + 1e-6)
mndwi = (green - swir1) / (green + swir1 + 1e-6)
ndmri = (swir1 - nir) / (swir1 + nir + 1e-6)

# Mặt nạ kim loại thô
metal_mask = (
    (swir1 > 0.20)
    & (ndmri > 0.08)
    & (ndvi < 0.22)
    & (mndwi < -0.05)
    & commune_mask
)

# 4. BỘ LỌC ĐỘ ĐỒNG NHẤT BỀ MẶT (Texture Variance Filter)
# Nhà xưởng có bề mặt SWIR1 rất mịn; nhà dân có SWIR1 biến thiên rất mạnh do chênh cốt sàn
swir1_norm = stretch_band(swir1)
# Tính độ lệch chuẩn trong cửa sổ 3x3 pixel (~60m x 60m)
local_std = generic_filter(swir1_norm, np.std, size=3)

# Chỉ lấy những nơi là kim loại VÀ bề mặt đồng nhất (độ lệch chuẩn thấp)
# Ngưỡng std < 0.08 loại bỏ hầu hết nhà ống nhấp nhô
smooth_metal_mask = metal_mask & (local_std < 0.08)

# 5. BỘ LỌC HÌNH THÁI HỌC VÀ ĐỘ ĐẶC HÌNH HỌC (Solidity Filter)
MIN_PIXELS = 8  # ~3.000 m2
labeled = label(smooth_metal_mask)
regions = regionprops(labeled)

clean_mask = np.zeros_like(metal_mask, dtype=bool)
final_boxes = []

for r in regions:
  # r.solidity: Tỷ lệ diện tích thực / Diện tích bao lồi (Convex Hull)
  # Nhà xưởng hình chữ nhật đặc thường có solidity > 0.70
  # Cụm nhà dân zíc zắc theo ngõ ngách thường có solidity < 0.60
  if r.area >= MIN_PIXELS and r.solidity >= 0.68:
    clean_mask[labeled == r.label] = True
    minr, minc, maxr, maxc = r.bbox
    final_boxes.append((minr, minc, maxr, maxc, r.area, r.solidity))

# 6. Thống kê lại
pixel_area_m2 = 19.3 * 19.3
total_pixels = np.sum(clean_mask)
total_ha = (total_pixels * pixel_area_m2) / 10000.0

print("=" * 65)
print(
    f"[*] Số cụm nhà xưởng CHUẨN XÁC sau lọc kết cấu: {len(final_boxes)} cụm"
)
print(f"[+] Tổng số pixel nhà xưởng thực tế:           {total_pixels:,} pixels")
print(f"[+] Diện tích nhà xưởng sau tinh chỉnh:        {total_ha:.2f} ha")
print("=" * 65)

# 7. Trực quan hóa
r_norm = stretch_band(red)
g_norm = stretch_band(green)
b_norm = stretch_band(blue)
rgb_img = np.dstack((r_norm, g_norm, b_norm))
rgb_img[~commune_mask] = 0

fig, ax = plt.subplots(1, 2, figsize=(16, 8), dpi=150)

# Bản đồ độ nhấp nhô kết cấu (Local Std) - Trắng là nhà dân gồ ghề, Tối là xưởng phẳng
ax[0].imshow(local_std, cmap="viridis")
ax[0].set_title(
    "1. Bản đồ kết cấu bề mặt (Local Variance)\n(Nhà dân nhấp nhô sáng màu,"
    " Nhà xưởng tối màu)",
    fontsize=11,
    fontweight="bold",
)
ax[0].axis("off")

# Kết quả sau lọc
ax[1].imshow(rgb_img)
for minr, minc, maxr, maxc, area, sol in final_boxes:
  rect = mpatches.Rectangle(
      (minc, minr),
      maxc - minc,
      maxr - minr,
      fill=False,
      edgecolor="#00FF00",
      linewidth=2,
  )
  ax[1].add_patch(rect)

overlay = rgb_img.copy()
overlay[clean_mask] = [1.0, 0.0, 0.0]
ax[1].imshow(overlay, alpha=0.5)
ax[1].set_title(
    f"2. Cụm nhà xưởng sau khi khử nhà dân (Còn {len(final_boxes)} cụm -"
    f" {total_ha:.2f} ha)",
    fontsize=11,
    fontweight="bold",
)
ax[1].axis("off")

plt.tight_layout()
plt.show()

7. Mặt nước tự nhiên và hệ thống nuôi trồng thủy sản:

Sông chính, kênh rạch lớn (rộng  >20m ), hồ thủy lợi, hồ chứa nước đô thị.

Hệ thống ao/đầm nuôi tôm cá dạng ô bàn cờ (mỗi ô thường  >500−1.000m2 ). Sự kết hợp giữa kênh Green và SWIR (chỉ số MNDWI) cho phép bóc tách ranh giới bờ đập và mặt nước rất sắc.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects

# 1. Đường dẫn file ảnh
image_path = "/content/s2_multiband_VNM_27_1_1_1_2020_03.tif"


def stretch_band(band, lower_percent=2, upper_percent=98):
  valid = band[~np.isnan(band) & (band > 0)]
  if len(valid) == 0:
    return band
  p_low, p_high = np.percentile(valid, (lower_percent, upper_percent))
  return np.clip((band - p_low) / (p_high - p_low + 1e-6), 0, 1)


# 2. Đọc kênh ảnh
with rasterio.open(image_path) as src:
  blue = src.read(1).astype(float)
  green = src.read(2).astype(float)
  red = src.read(3).astype(float)
  nir = src.read(4).astype(float)
  swir1 = src.read(5).astype(float)

  # Đảm bảo chuẩn hóa thang đo nếu ảnh đang ở dạng số nguyên 0 - 10000
  if np.nanmax(red) > 10.0:
    blue /= 10000.0
    green /= 10000.0
    red /= 10000.0
    nir /= 10000.0
    swir1 /= 10000.0

  if src.nodata is not None:
    commune_mask = red != src.nodata
  else:
    commune_mask = (red > 0) | (green > 0) | (blue > 0)

# 3. Tính MNDWI & NDVI chuẩn
mndwi = (green - swir1) / (green + swir1 + 1e-6)
ndvi = (nir - red) / (nir + red + 1e-6)

# KHẮC PHỤC LỖI:
# - Nâng nhẹ ngưỡng MNDWI từ 0.05 lên 0.12 để cắt bỏ bê tông/mái tôn ướt
# - green > 0.04: Cắt bỏ hoàn toàn bóng râm tòa nhà cao tầng
# - swir1 < 0.10: Mặt nước bắt buộc phải hấp thụ mạnh dải hồng ngoại ngắn
water_mask = (
    (mndwi > 0.12)
    & (ndvi < 0.15)
    & (green > 0.04)
    & (swir1 < 0.10)
    & commune_mask
)

# Nâng min_size lên 4 pixels (~1.500 m2) để loại sạch đốm nhiễu ngõ ngách
clean_water = remove_small_objects(water_mask, min_size=4)

# 4. Phân loại hình thái mặt nước chuẩn xác hơn
labeled_water = label(clean_water)
regions = regionprops(labeled_water)

natural_water_mask = np.zeros_like(clean_water, dtype=bool)
aquaculture_mask = np.zeros_like(clean_water, dtype=bool)

pixel_area_m2 = 19.3 * 19.3

for r in regions:
  area_m2 = r.area * pixel_area_m2
  aspect_ratio = r.major_axis_length / (r.minor_axis_length + 1e-6)

  # Hồ chứa lớn (> 1.2 ha) HOẶC Sông ngòi thực sự (phải dài VÀ diện tích tối thiểu > 0.8 ha)
  if area_m2 >= 12000 or (aspect_ratio >= 4.0 and area_m2 >= 8000):
    natural_water_mask[labeled_water == r.label] = True
  else:
    aquaculture_mask[labeled_water == r.label] = True

# 5. Thống kê
total_water_ha = np.sum(clean_water) * pixel_area_m2 / 10000.0
natural_ha = np.sum(natural_water_mask) * pixel_area_m2 / 10000.0
aqua_ha = np.sum(aquaculture_mask) * pixel_area_m2 / 10000.0

print("=" * 65)
print(f"[+] Tổng diện tích mặt nước thực tế:          {total_water_ha:.2f} ha")
print(f"[*] Diện tích sông chính / hồ điều hòa lớn:   {natural_ha:.2f} ha")
print(f"[*] Diện tích ao hồ phân tán / đầm thủy sản:  {aqua_ha:.2f} ha")
print("=" * 65)

# 6. Trực quan hóa kết quả
r_norm = stretch_band(red)
g_norm = stretch_band(green)
b_norm = stretch_band(blue)
rgb_img = np.dstack((r_norm, g_norm, b_norm))
rgb_img[~commune_mask] = 0

fig, ax = plt.subplots(1, 2, figsize=(16, 8), dpi=150)

ax[0].imshow(rgb_img)
ax[0].set_title("1. Ảnh RGB Màu Thực", fontsize=11, fontweight="bold")
ax[0].axis("off")

overlay = rgb_img.copy()
# Xanh dương cho sông/hồ lớn
overlay[natural_water_mask] = [0.0, 0.4, 1.0]
# Xanh lơ (cyan) cho ao hồ nhỏ
overlay[aquaculture_mask] = [0.0, 1.0, 0.9]

ax[1].imshow(overlay)
ax[1].set_title(
    f"2. Bóc tách mặt nước chuẩn ({total_water_ha:.2f} ha)\n(Sông/Hồ lớn:"
    f" {natural_ha:.2f} ha | Ao nhỏ: {aqua_ha:.2f} ha)",
    fontsize=11,
    fontweight="bold",
)
ax[1].axis("off")

plt.tight_layout()
plt.show()

8. Lấy 1 ảnh tif, hiện thị ảnh google map của nó và kèm theo link

In [ ]:
!pip install contextily rasterio

In [ ]:
import contextily as cx
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from pyproj import Transformer

tif_path = "/content/s2_multiband_VNM_27_1_1_1_2020_03.tif"

# 1. Đọc toạ độ và thông số không gian từ file TIF
with rasterio.open(tif_path) as src:
    bounds = src.bounds  # left (min_lon), bottom (min_lat), right (max_lon), top (max_lat)
    crs = src.crs

min_lon, min_lat = bounds.left, bounds.bottom
max_lon, max_lat = bounds.right, bounds.top
center_lon = (min_lon + max_lon) / 2.0
center_lat = (min_lat + max_lat) / 2.0

# In chi tiết tọa độ thực tế tương ứng trên Google Maps
print("=" * 80)
print("[+] THÔNG TIN TỌA ĐỘ ĐỊA LÝ (WGS84 / GOOGLE MAPS):")
print(f"    - Góc Tây Bắc (Tây - Bắc): {max_lat:.6f}° N, {min_lon:.6f}° E")
print(f"    - Góc Đông Bắc (Đông - Bắc): {max_lat:.6f}° N, {max_lon:.6f}° E")
print(f"    - Góc Tây Nam  (Tây - Nam) : {min_lat:.6f}° N, {min_lon:.6f}° E")
print(f"    - Góc Đông Nam (Đông - Nam): {min_lat:.6f}° N, {max_lon:.6f}° E")
print(f"    - Tọa độ tâm điểm (Center) : {center_lat:.6f}° N, {center_lon:.6f}° E")
print(f"    👉 Đường link Google Maps  : https://www.google.com/maps?q={center_lat:.6f},{center_lon:.6f}")
print("=" * 80)

# 2. Tải ảnh vệ tinh Google Maps khớp theo khung bao
google_sat_url = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"
img_raw, ext = cx.bounds2img(
    min_lon,
    min_lat,
    max_lon,
    max_lat,
    zoom=15,
    ll=True,
    source=google_sat_url
)

# 3. Chuyển toạ độ sang EPSG:3857 để cắt gọt chuẩn xác theo extent TIF
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
min_x_3857, min_y_3857 = transformer.transform(min_lon, min_lat)
max_x_3857, max_y_3857 = transformer.transform(max_lon, max_lat)

ext_xmin, ext_xmax, ext_ymin, ext_ymax = ext
img_h, img_w, _ = img_raw.shape

col_start = int(np.clip((min_x_3857 - ext_xmin) / (ext_xmax - ext_xmin) * img_w, 0, img_w))
col_end   = int(np.clip((max_x_3857 - ext_xmin) / (ext_xmax - ext_xmin) * img_w, 0, img_w))
row_start = int(np.clip((ext_ymax - max_y_3857) / (ext_ymax - ext_ymin) * img_h, 0, img_h))
row_end   = int(np.clip((ext_ymax - min_y_3857) / (ext_ymax - ext_ymin) * img_h, 0, img_h))

img_cropped = img_raw[row_start:row_end, col_start:col_end]

# 4. Đọc kênh RGB từ file TIF gốc
with rasterio.open(tif_path) as src:
    r = src.read(3).astype(float)
    g = src.read(2).astype(float)
    b = src.read(1).astype(float)

def stretch(band):
    valid = band[~np.isnan(band) & (band > 0)]
    if len(valid) == 0: return band
    p2, p98 = np.percentile(valid, (2, 98))
    return np.clip((band - p2) / (p98 - p2 + 1e-6), 0, 1)

rgb = np.dstack((stretch(r), stretch(g), stretch(b)))

# 5. Hiển thị ảnh kèm trục kinh độ/vĩ độ
fig, ax = plt.subplots(1, 2, figsize=(16, 8), dpi=150)

# Khung toạ độ hiển thị (left, right, bottom, top)
extent_wgs84 = [min_lon, max_lon, min_lat, max_lat]

# Ảnh 1: Google Maps
ax[0].imshow(img_cropped, extent=extent_wgs84)
ax[0].set_title("1. Google Maps vệ tinh (Đã crop khớp 100%)", fontweight="bold")
ax[0].set_xlabel("Kinh độ (°E)")
ax[0].set_ylabel("Vĩ độ (°N)")
ax[0].grid(True, linestyle="--", alpha=0.5, color="yellow")

# Ảnh 2: Sentinel-2 TIF
ax[1].imshow(rgb, extent=extent_wgs84)
ax[1].set_title("2. Ảnh Sentinel-2 TIF gốc (20m/pixel)", fontweight="bold")
ax[1].set_xlabel("Kinh độ (°E)")
ax[1].grid(True, linestyle="--", alpha=0.5, color="white")

plt.tight_layout()
plt.show()

Với thông số kỹ thuật cụ thể của tệp ảnh:

* **Độ phân giải không gian (Spatial Resolution):** $\sim 19.3\text{m} \times 19.3\text{m}$ (mỗi pixel tương đương diện tích mặt đất khoảng $\approx 372\text{ m}^2$).
* **Kích thước khung hình:** $157 \times 147$ pixels (tổng diện tích bao phủ khoảng $3\text{ km} \times 2.8\text{ km} \approx 8.5\text{ km}^2$, trong đó diện tích thực của xã khoảng $4.07\text{ km}^2$).
* **Độ sâu quang phổ:** **10 kênh phổ (Bands)** gồm đầy đủ dải ánh sáng nhìn thấy (RGB), Cận hồng ngoại (NIR), và Hồng ngoại sóng ngắn (SWIR1, SWIR2).

Khả năng nhận diện đối tượng (Object Detection / Semantic Segmentation) được phân định cụ thể theo kích thước hình học và đặc tính phản xạ phổ như sau:

---

### 1. Các đối tượng **DETECT ĐƯỢC RẤT RÕ NÉT VÀ CHUẨN XÁC**

Đây là những thực thể nhân tạo hoặc tự nhiên có kích thước bề mặt tối thiểu từ **vài trăm mét vuông trở lên** (chiếm từ $2 - 3$ pixel liền kề):

* **Khu công nghiệp, nhà máy, kho bãi logistics quy mô lớn:**
* Các nhà xưởng thường có diện tích từ $1.000\text{ m}^2$ đến hàng chục nghìn $\text{m}^2$ (tương đương $5 - 50+$ pixel).
* **Kênh nhận diện:** Kênh **SWIR1 và SWIR2** cực kỳ nhạy với vật liệu kim loại (mái tôn lợp xưởng) và bề mặt bê tông công nghiệp, tạo nên tín hiệu phản xạ trắng/sáng rõ rệt tách biệt hoàn toàn khỏi khu dân cư thông thường.


* **Mạng lưới giao thông trục chính:**
* Đường cao tốc, quốc lộ, đại lộ đô thị hoặc đường liên huyện có lộ giới mặt đường $> 15 - 20\text{m}$ (chiếm bề ngang khoảng $1$ pixel và kéo dài liên tục thành vệt).
* Nút giao thông khác mức, cầu vượt sông lớn, cảng bốc dỡ ven sông.


* **Mặt nước tự nhiên và hệ thống nuôi trồng thủy sản:**
* Sông chính, kênh rạch lớn (rộng $> 20\text{m}$), hồ thủy lợi, hồ chứa nước đô thị.
* Hệ thống ao/đầm nuôi tôm cá dạng ô bàn cờ (mỗi ô thường $> 500 - 1.000\text{m}^2$). Sự kết hợp giữa kênh **Green và SWIR (chỉ số MNDWI)** cho phép bóc tách ranh giới bờ đập và mặt nước rất sắc.


* **Khối công trình công cộng & hạ tầng đô thị tập trung:**
* Bệnh viện tuyến huyện/tỉnh, trường học cấp 3/đại học (khối nhà nhiều tầng kèm sân tập), sân vận động, trung tâm hành chính xã/huyện.
* Các cụm chung cư, khối nhà cao tầng khu đô thị mới.


* **Mảng cấu trúc nông nghiệp & tự nhiên quy mô lớn:**
* Cánh đồng lúa tập trung, vùng chuyên canh hoa màu, trang trại trồng cây ăn trái/cây công nghiệp (cao su, cà phê).
* Đồi rừng, công viên tập trung, dải cây xanh ven sông (nhờ tín hiệu phản xạ cực mạnh từ kênh **NIR/NDVI**).


* **Vùng đất đang giải phóng mặt bằng / Công trường san nền:**
* Các khu đất trống bị cạo lớp phủ thực vật để lộ nền cát/đất đỏ chuẩn bị xây dựng (kênh **BSI/SWIR** bắt tín hiệu đất trơ rực sáng).



---

### 2. Các đối tượng **DETECT DẠNG CỤM (Clustered Pattern), KHÔNG THỂ TÁCH RỜI ĐƠN LẺ**

Do 1 pixel rộng $\approx 20\text{m}$, những đối tượng này bị gộp chung vào một điểm ảnh (Mixed Pixel):

* **Nhà ở dân cư nông thôn hoặc nhà ống đô thị:**
* Một ngôi nhà ống $5\text{m} \times 20\text{m}$ ($100\text{m}^2$) sẽ bị gộp chung với sân, mái hiên, ngõ đi hoặc tán cây bên cạnh vào cùng 1 pixel.
* **Khả năng nhận diện:** Mô hình không đếm được "xã này có 1.500 ngôi nhà", nhưng **phát hiện được toàn bộ khu vực lõi dân cư tập trung** và đo được mật độ xây dựng trung bình ($\% \text{Built-up Density}$).


* **Đường ngõ xóm, đường nội bộ:**
* Đường làng, ngõ ngách rộng $2 - 5\text{m}$ không thể nhìn thấy trực tiếp thành đường kẻ, nhưng làm biến đổi giá trị phản xạ của pixel xung quanh (tạo ra hỗn hợp đất/bê tông xen kẽ).



---

### 3. Các đối tượng **HOÀN TOÀN KHÔNG THỂ DETECT ĐƯỢC** (Vượt quá giới hạn vật lý)

Tuyệt đối không sử dụng ảnh này cho các bài toán nhận diện đối tượng vi mô (như huấn luyện YOLO tìm vật thể):

* Xe ô tô, xe máy, container di chuyển trên đường.
* Từng cây bóng mát đơn lẻ trên vỉa hè.
* Cột điện, trạm biến áp nhỏ, biển hiệu quảng cáo.
* Con người, gia súc ngoài đồng ruộng.

---

### 4. Tóm lược tác vụ khả thi nhất trên ảnh này

Với ảnh đa phổ kích thước này, bài toán khả thi và mang lại giá trị cao nhất là **Phân vùng ngữ nghĩa (Semantic Segmentation)** bằng các mô hình như **U-Net, DeepLabV3+ hoặc SegFormer**. Bạn có thể chia bức ảnh thành **5 lớp đối tượng bề mặt (Masks)**:

1. *Lớp Đất công nghiệp/xây dựng lớn* (Nhà xưởng, kho, công trình kiên cố).
2. *Lớp Khu dân cư/đô thị nén* (Nhà ở tập trung, ngõ xóm).
3. *Lớp Đất nông nghiệp/cây xanh* (Đồng lúa, vườn, rừng).
4. *Lớp Mặt nước* (Sông, ao nuôi thủy sản).
5. *Lớp Đất trống/công trường xây dựng* (Đang san lấp).

Tỷ lệ diện tích và cấu trúc không gian của 5 lớp này chính là các đặc trưng định lượng mạnh nhất để đưa vào mô hình dự báo kinh tế - xã hội.